# 11 · File `.plt` giống hệt nhau giữa các user

Một số file `.plt` xuất hiện **giống hệt từng byte** ở thư mục của hai hay nhiều user. Giống hệt từng byte nghĩa là **cùng một log của một thiết bị**: hai máy đi cùng nhau vẫn khác nhau, vì nhiễu GPS khoảng 34 m (notebook 01) và giờ bật máy khác nhau.

**Câu hỏi:** vì sao file bị chép sang user khác, và Home/Office có cần xử lý gì không?
- **Giả thuyết A, đi cùng nhau:** các user đi cùng nhau, và một người chia sẻ log cho người kia. Chuyến đi khi đó đúng là của cả hai, nên **giữ nguyên**.
- **Giả thuyết B, chép nhầm:** log của người này bị gán cho người kia. Khi đó chỗ ở của một người có thể bị gán sai cho người kia.

**Bằng chứng, chỉ dùng điểm GPS thật:**
1. **Thống kê:** bao nhiêu file, cặp user nào, chiếm bao nhiêu phần dữ liệu của mỗi user.
2. **Chỗ quen:** file chung có nằm ở những nơi mà **từng user** đã đến trong **file riêng** của họ không? So với mức nền là một user ngẫu nhiên không liên quan.
3. **Ghi riêng mà vẫn ở cạnh nhau:** khi hai user cùng ghi bằng **hai máy riêng** vào cùng thời điểm, họ có ở gần nhau không? Nếu có nhiều hơn hẳn so với các cặp không có file chung, thì họ thật sự hay đi cùng nhau.
4. **Tác động:** bao nhiêu stay-point và thời gian ở lại của mỗi user nằm trong file chung.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

NOTEBOOK = "11_shared_recordings"
DATA_DIR = Path("../data/raw")
OUT_DIR = Path("outputs") / NOTEBOOK
FIG_DIR, TABLE_DIR, CACHE_DIR, MAP_DIR = (OUT_DIR / d for d in ("figures", "tables", "cache", "maps"))
for d in (FIG_DIR, TABLE_DIR, CACHE_DIR, MAP_DIR):
    d.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(Path("../src").resolve()))
import hashlib
from itertools import combinations

from sklearn.neighbors import BallTree

from gps.data.processing import _haversine_vectorized

USERS_DIR = Path("../data/processed/users")
STAY_DIR = Path("../data/processed/staypoints")
NEAR_M = 200                 # cùng bán kính stay-point
EARTH_M = 6_371_008.8
rng = np.random.default_rng(20261005)

## 1. Thống kê file chung
Mỗi file `.plt` được băm MD5 theo nội dung. Thời gian và số điểm lấy từ dữ liệu đã làm sạch (`data/processed/users`).

In [2]:
FILES_PATH = CACHE_DIR / "files.parquet"
if FILES_PATH.exists():
    files = pd.read_parquet(FILES_PATH)
else:
    rows = [{"user_id": p.parts[-3], "file": p.name, "md5": hashlib.md5(p.read_bytes()).hexdigest()}
            for p in sorted(DATA_DIR.glob("*/Trajectory/*.plt"))]
    bounds = []
    for path in sorted(USERS_DIR.glob("user_*.parquet")):
        d = pd.read_parquet(path, columns=["source_file", "datetime"])
        b = d.groupby("source_file")["datetime"].agg(start="min", end="max", n_points="size").reset_index()
        bounds.append(b.rename(columns={"source_file": "file"}).assign(user_id=path.stem.split("_")[1]))
    files = pd.DataFrame(rows).merge(pd.concat(bounds), on=["user_id", "file"], how="left")
    files.to_parquet(FILES_PATH)

owners = files.groupby("md5")["user_id"].nunique()
files["shared"] = files["md5"].map(owners) > 1
shared = files[files["shared"]]
print(f"Files: {len(files):,} | contents held by > 1 user: {shared['md5'].nunique():,} | extra copies: "
      f"{len(shared) - shared['md5'].nunique():,} | users involved: {shared['user_id'].nunique()}")
print(f"Clean points in shared files: {shared['n_points'].sum():,.0f} of {files['n_points'].sum():,.0f} "
      f"({shared['n_points'].sum() / files['n_points'].sum():.1%}); redundant copies: "
      f"{(shared['n_points'].sum() - shared.drop_duplicates('md5')['n_points'].sum()):,.0f}")
print("Same file name in every copy:", bool(shared.groupby("md5")["file"].nunique().eq(1).all()),
      "| same content twice inside one user:", int(files.groupby(["user_id", "md5"]).size().gt(1).sum()))

pair_rows = []
per_user_pts = files.groupby("user_id")["n_points"].sum()
for md5, g in shared.groupby("md5"):
    for a, b in combinations(sorted(g["user_id"]), 2):
        pair_rows.append({"user_a": a, "user_b": b, "md5": md5, "n_points": g["n_points"].iloc[0]})
pairs = (pd.DataFrame(pair_rows).groupby(["user_a", "user_b"])
         .agg(shared_files=("md5", "size"), shared_points=("n_points", "sum")).reset_index())
pairs["share_of_a"] = pairs["shared_points"] / pairs["user_a"].map(per_user_pts)
pairs["share_of_b"] = pairs["shared_points"] / pairs["user_b"].map(per_user_pts)
pairs = pairs.sort_values("shared_files", ascending=False, ignore_index=True)
pairs.to_csv(TABLE_DIR / "shared_pairs.csv", index=False)
print(f"User pairs: {len(pairs)}; groups of 3+ users sharing a file: {int((owners > 2).sum())} contents")
pairs.head(15).round(3)

Files: 18,670 | contents held by > 1 user: 821 | extra copies: 856 | users involved: 52
Clean points in shared files: 2,913,206 of 24,177,459 (12.0%); redundant copies: 1,463,229
Same file name in every copy: True | same content twice inside one user: 0
User pairs: 44; groups of 3+ users sharing a file: 33 contents


,user_a,user_b,shared_files,shared_points,share_of_a,share_of_b
0,153,163,225,612026,0.284,0.632
1,126,167,145,199587,0.485,0.328
2,000,003,129,160928,0.926,0.332
3,011,088,59,27030,0.298,1.000
4,128,153,54,38993,0.032,0.018
5,089,144,25,23395,0.757,0.040
6,057,150,22,2793,1.000,1.000
7,094,150,22,2793,1.000,1.000
8,057,094,22,2793,1.000,1.000
9,081,125,22,32704,0.739,0.361


### 1.1 File chung chiếm bao nhiêu phần dữ liệu của mỗi user?
Nếu toàn bộ dữ liệu của một user nằm trong file chung, thì Home/Office của user đó hoàn toàn dựa trên log của một thiết bị khác.

In [3]:
by_user = files.groupby("user_id").agg(files=("file", "size"), shared_files=("shared", "sum"), points=("n_points", "sum"))
by_user["shared_points"] = shared.groupby("user_id")["n_points"].sum()
by_user = by_user.fillna({"shared_points": 0})
by_user["shared_share"] = by_user["shared_points"] / by_user["points"]
involved = by_user[by_user["shared_files"] > 0].sort_values("shared_share", ascending=False)
involved.to_csv(TABLE_DIR / "shared_share_by_user.csv")
print("Users by share of their clean points in shared files:",
      pd.cut(involved["shared_share"], [0, 0.1, 0.25, 0.5, 0.9, 1.0], include_lowest=True).value_counts().sort_index().to_dict())
involved.head(12).round(3)

Users by share of their clean points in shared files: {Interval(-0.001, 0.1, closed='right'): 13, Interval(0.1, 0.25, closed='right'): 7, Interval(0.25, 0.5, closed='right'): 11, Interval(0.5, 0.9, closed='right'): 4, Interval(0.9, 1.0, closed='right'): 17}


,files,shared_files,points,shared_points,shared_share
user_id,,,,,
059,8,8,12606,12606.0,1.000
070,11,11,35311,35311.0,1.000
057,22,22,2793,2793.0,1.000
088,59,59,27030,27030.0,1.000
127,10,10,22853,22853.0,1.000
150,22,22,2793,2793.0,1.000
156,2,2,16277,16277.0,1.000
124,10,10,120111,120111.0,1.000
108,9,9,484,484.0,1.000


## 2. File chung nằm ở chỗ quen của ai?
Với mỗi user, "chỗ quen" là mọi điểm GPS trong **file riêng** của user đó (file không chung với ai). Một file chung được tính là **quen với user X** khi ít nhất 50% số điểm của nó nằm trong 200 m quanh một chỗ quen của X.

**Mức nền:** nhiều user GeoLife cùng sống và làm việc quanh một khu (Hải Điến, Bắc Kinh), nên một file bất kỳ cũng có thể "quen" với một người lạ. Vì vậy mỗi file chung còn được đo với **3 user ngẫu nhiên** không có trong nhóm chia sẻ file đó.

In [4]:
_trees: dict[str, BallTree | None] = {}


def own_tree(uid: str) -> BallTree | None:
    """BallTree (haversine) trên điểm của các file riêng của user, làm tròn ~10 m để gọn."""
    if uid not in _trees:
        d = pd.read_parquet(USERS_DIR / f"user_{uid}.parquet", columns=["source_file", "lat", "lon"])
        own = set(files.loc[(files["user_id"] == uid) & ~files["shared"], "file"])
        xy = d.loc[d["source_file"].isin(own), ["lat", "lon"]].round(4).drop_duplicates().to_numpy()
        if len(_trees) > 40:          # giới hạn bộ nhớ
            _trees.clear()
        _trees[uid] = BallTree(np.radians(xy), metric="haversine") if len(xy) else None
    return _trees[uid]


_points: dict[tuple[str, str], np.ndarray] = {}


def file_points(uid: str, fname: str) -> np.ndarray:
    key = (uid, fname)
    if key not in _points:
        d = pd.read_parquet(USERS_DIR / f"user_{uid}.parquet", columns=["source_file", "lat", "lon"])
        for f, g in d.groupby("source_file"):
            xy = g[["lat", "lon"]].to_numpy()
            _points[(uid, f)] = xy[:: max(1, len(xy) // 300)]
    return _points.get(key, np.empty((0, 2)))


def familiarity(xy: np.ndarray, uid: str) -> float:
    tree = own_tree(uid)
    if tree is None or len(xy) == 0:
        return np.nan
    dist, _ = tree.query(np.radians(xy), k=1)
    return float((dist[:, 0] * EARTH_M <= NEAR_M).mean())


FAM_PATH = CACHE_DIR / "familiarity.parquet"
if FAM_PATH.exists():
    fam = pd.read_parquet(FAM_PATH)
else:
    all_users = sorted(files["user_id"].unique())
    rows = []
    for md5, g in shared.sort_values("user_id").groupby("md5"):
        members = list(g["user_id"])
        xy = file_points(members[0], g["file"].iloc[0])
        strangers = rng.choice([u for u in all_users if u not in members], size=3, replace=False)
        for uid in members:
            rows.append({"md5": md5, "user_id": uid, "role": "member", "familiar": familiarity(xy, uid)})
        for uid in strangers:
            rows.append({"md5": md5, "user_id": uid, "role": "stranger", "familiar": familiarity(xy, uid)})
    fam = pd.DataFrame(rows)
    fam.to_parquet(FAM_PATH)
# NaN = user không có file riêng nào (toàn bộ dữ liệu nằm trong file chung): không đo được, tách riêng
fam["is_familiar"] = fam["familiar"] >= 0.5
measurable = fam.dropna(subset=["familiar"])
print("Share of (file, user) with >= 50% of the file's points near the user's own places (users with own files only):")
print(measurable.groupby("role")["is_familiar"].agg(["mean", "size"]).round(3).to_string())

members = fam[fam["role"] == "member"]
pattern = members.groupby("md5").agg(familiar_members=("is_familiar", "sum"), measurable=("familiar", "count"),
                                     members=("familiar", "size"))
pattern["pattern"] = np.select(
    [pattern["measurable"] == 0, pattern["familiar_members"] == pattern["measurable"], pattern["familiar_members"] == 0],
    ["no member has own files", "familiar to all measurable members", "familiar to no member"],
    "familiar to some members only")
fam_table = pattern["pattern"].value_counts().to_frame("files")
fam_table.to_csv(TABLE_DIR / "shared_file_familiarity.csv")
fam_table

Share of (file, user) with >= 50% of the file's points near the user's own places (users with own files only):
           mean  size
role                 
member    0.755  1498
stranger  0.234  2312


,files
pattern,
familiar to all measurable members,558
familiar to no member,170
familiar to some members only,71
no member has own files,22


### 2.1 File chung không quen với ai là gì?
So sánh vị trí (khoảng cách tới Hải Điến, nơi phần lớn user GeoLife sống và làm việc), múi giờ và độ dài giữa file chung **không quen với user nào** và file chung **quen với mọi user đo được**. Kèm theo: user nào trong các nhóm lớn có file `labels.txt`.

In [5]:
first_copy = shared.sort_values("user_id").drop_duplicates("md5").set_index("md5")
rows = []
for md5, kind in pattern["pattern"].items():
    if kind not in ("familiar to no member", "familiar to all measurable members"):
        continue
    r = first_copy.loc[md5]
    xy = file_points(r["user_id"], r["file"])
    if len(xy) == 0:
        continue
    lat, lon = np.median(xy[:, 0]), np.median(xy[:, 1])
    rows.append({"pattern": kind, "km_from_haidian": _haversine_vectorized(np.array([lat]), np.array([lon]),
                                                                           np.array([39.99]), np.array([116.33]))[0] / 1000,
                 "hours": (r["end"] - r["start"]).total_seconds() / 3600})
where = pd.DataFrame(rows)
where_table = where.groupby("pattern").agg(files=("hours", "size"), median_km_from_haidian=("km_from_haidian", "median"),
                                           over_50_km=("km_from_haidian", lambda s: int((s > 50).sum())),
                                           median_hours=("hours", "median"))
where_table.to_csv(TABLE_DIR / "shared_file_location.csv")
has_labels = {u: (DATA_DIR / u / "labels.txt").exists() for u in ["057", "094", "150", "011", "088", "055", "075", "153", "163"]}
print("labels.txt present:", has_labels)
where_table.round(1)

labels.txt present: {'057': False, '094': False, '150': False, '011': False, '088': True, '055': False, '075': True, '153': True, '163': True}


,files,median_km_from_haidian,over_50_km,median_hours
pattern,,,,
familiar to all measurable members,558,2.1,6,0.6
familiar to no member,170,806.8,110,4.9


## 3. Ghi bằng hai máy riêng mà vẫn ở cạnh nhau?
Đây là phép thử trực tiếp nhất cho giả thuyết A. Chỉ dùng **file riêng** của mỗi user, nên mọi điểm đều đến từ hai thiết bị khác nhau. Mỗi user được thu về tối đa 1 điểm mỗi phút. Với mỗi phút mà cả hai user đều có điểm, đo khoảng cách giữa họ.

- `matched_minutes`: số phút cả hai cùng có điểm;
- `close_share`: tỉ lệ số phút đó mà hai người cách nhau ≤ 200 m.

So sánh các cặp **có file chung** với các cặp **không có file chung** (mức nền), chỉ xét các cặp có ít nhất 30 phút trùng nhau.

In [6]:
CO_PATH = CACHE_DIR / "colocation.parquet"
if CO_PATH.exists():
    co = pd.read_parquet(CO_PATH)
else:
    per_minute = []
    for path in sorted(USERS_DIR.glob("user_*.parquet")):
        uid = path.stem.split("_")[1]
        own = set(files.loc[(files["user_id"] == uid) & ~files["shared"], "file"])
        d = pd.read_parquet(path, columns=["source_file", "datetime", "lat", "lon"])
        d = d[d["source_file"].isin(own)]
        d = d.assign(minute=d["datetime"].dt.floor("min")).drop_duplicates("minute")
        per_minute.append(d[["minute", "lat", "lon"]].assign(user_id=uid))
    pm = pd.concat(per_minute, ignore_index=True)
    busy = pm.groupby("minute")["user_id"].transform("size") > 1
    j = pm[busy].merge(pm[busy], on="minute", suffixes=("_a", "_b"))
    j = j[j["user_id_a"] < j["user_id_b"]]
    j["dist_m"] = _haversine_vectorized(j["lat_a"].to_numpy(), j["lon_a"].to_numpy(), j["lat_b"].to_numpy(), j["lon_b"].to_numpy())
    co = (j.assign(close=j["dist_m"] <= NEAR_M).groupby(["user_id_a", "user_id_b"])
          .agg(matched_minutes=("close", "size"), close_minutes=("close", "sum")).reset_index())
    co.to_parquet(CO_PATH)
co["close_share"] = co["close_minutes"] / co["matched_minutes"]
pair_keys = set(zip(pairs["user_a"], pairs["user_b"]))
co["has_shared_files"] = [k in pair_keys for k in zip(co["user_id_a"], co["user_id_b"])]
enough = co[co["matched_minutes"] >= 30]
summary = enough.groupby("has_shared_files").agg(pairs=("close_share", "size"), median_close_share=("close_share", "median"),
                                                 share_of_pairs_close_half_the_time=("close_share", lambda s: (s >= 0.5).mean()),
                                                 total_close_minutes=("close_minutes", "sum"))
summary.to_csv(TABLE_DIR / "colocation_summary.csv")
print(f"Pairs with shared files: {len(pairs)}, of which with >= 30 independent overlapping minutes: "
      f"{int(enough['has_shared_files'].sum())}")
summary.round(3)

Pairs with shared files: 44, of which with >= 30 independent overlapping minutes: 18


,pairs,median_close_share,share_of_pairs_close_half_the_time,total_close_minutes
has_shared_files,,,,
False,1596,0.000,0.011,16812
True,18,0.113,0.389,7428


In [7]:
shared_co = pairs.merge(co, left_on=["user_a", "user_b"], right_on=["user_id_a", "user_id_b"], how="left")
shared_co = shared_co[["user_a", "user_b", "shared_files", "share_of_a", "share_of_b", "matched_minutes", "close_minutes", "close_share"]]
shared_co.to_csv(TABLE_DIR / "shared_pairs_colocation.csv", index=False)
shared_co.round(3).head(20)

,user_a,user_b,shared_files,share_of_a,share_of_b,matched_minutes,close_minutes,close_share
0,153,163,225,0.284,0.632,3072.0,2698.0,0.878
1,126,167,145,0.485,0.328,3918.0,3589.0,0.916
2,000,003,129,0.926,0.332,167.0,2.0,0.012
3,011,088,59,0.298,1.000,NaN,NaN,NaN
4,128,153,54,0.032,0.018,5786.0,114.0,0.020
5,089,144,25,0.757,0.040,358.0,218.0,0.609
6,057,150,22,1.000,1.000,NaN,NaN,NaN
7,094,150,22,1.000,1.000,NaN,NaN,NaN
8,057,094,22,1.000,1.000,NaN,NaN,NaN
9,081,125,22,0.739,0.361,NaN,NaN,NaN


## 4. Tác động lên stay-point
Một stay-point được tính là **từ file chung** khi ít nhất 50% thời gian của nó nằm trong khoảng thời gian của một file chung của chính user đó.

In [8]:
rows = []
for path in sorted(STAY_DIR.glob("user_*.parquet")):
    uid = path.stem.split("_")[1]
    s = pd.read_parquet(path)
    if s.empty:
        continue
    sh = files[(files["user_id"] == uid) & files["shared"]].dropna(subset=["start"])
    inside = np.zeros(len(s))
    for t0, t1 in zip(sh["start"].to_numpy(), sh["end"].to_numpy(), strict=True):
        ov = (np.minimum(s["departure"].to_numpy(), t1) - np.maximum(s["arrival"].to_numpy(), t0)) / np.timedelta64(1, "m")
        inside = np.maximum(inside, np.clip(ov, 0, None))
    from_shared = inside >= 0.5 * s["duration_minutes"].to_numpy()
    rows.append({"user_id": uid, "stays": len(s), "stays_from_shared": int(from_shared.sum()),
                 "dwell_h": s["duration_minutes"].sum() / 60, "dwell_h_from_shared": s.loc[from_shared, "duration_minutes"].sum() / 60})
impact = pd.DataFrame(rows).set_index("user_id")
impact["dwell_share_from_shared"] = impact["dwell_h_from_shared"] / impact["dwell_h"]
impact.to_csv(TABLE_DIR / "staypoint_impact.csv")
print(f"Stay-points from shared files: {impact['stays_from_shared'].sum():,} of {impact['stays'].sum():,} "
      f"({impact['stays_from_shared'].sum() / impact['stays'].sum():.1%}); users with any: {(impact['stays_from_shared'] > 0).sum()}")
print("Users by share of dwell time from shared files:",
      pd.cut(impact["dwell_share_from_shared"], [0, 1e-9, 0.1, 0.5, 0.9, 1.0], include_lowest=True).value_counts().sort_index().to_dict())
impact.sort_values("dwell_share_from_shared", ascending=False).head(12).round(3)

Stay-points from shared files: 1,758 of 18,597 (9.5%); users with any: 48
Users by share of dwell time from shared files: {Interval(-0.001, 1e-09, closed='right'): 126, Interval(1e-09, 0.1, closed='right'): 27, Interval(0.1, 0.5, closed='right'): 18, Interval(0.5, 0.9, closed='right'): 1, Interval(0.9, 1.0, closed='right'): 2}


,stays,stays_from_shared,dwell_h,dwell_h_from_shared,dwell_share_from_shared
user_id,,,,,
124,6,6,8.044,8.044,1.000
156,7,7,7.577,7.577,1.000
075,13,12,71.108,60.260,0.847
055,23,12,130.744,60.260,0.461
070,13,11,52.229,21.123,0.404
129,11,2,10.455,4.148,0.397
000,405,291,1571.414,556.959,0.354
108,6,5,11.255,3.806,0.338
127,13,9,67.823,18.778,0.277


## Kết luận (tư liệu, không đổi quy tắc nào)

**Quyết định của dự án giữ nguyên: mỗi user là độc lập.** Dữ liệu trong thư mục của user nào được coi là của user đó, kể cả file chung. Notebook này chỉ ghi lại hiện tượng, các suy luận có thể có, và rủi ro cần nhớ ở các giai đoạn sau.

**Số liệu:**

| | Kết quả |
|---|---|
| Quy mô | 821 nội dung nằm ở > 1 user (856 bản sao thừa), 52 user, 44 cặp. 12.0% số điểm sạch nằm trong file chung, trong đó 1.46 triệu điểm là bản sao thừa. Bản sao luôn trùng cả tên file; không user nào có hai file giống nhau |
| User gần như không có dữ liệu riêng | 17 user có > 90% số điểm trong file chung; 11 user là 100% (057, 059, 070, 088, 094, 108, 109, 124, 127, 150, 156). Ba user **057, 094, 150 có toàn bộ dữ liệu giống hệt nhau**: cùng 22 file, không có nhãn |
| File chung nằm ở chỗ quen của ai? | 75.5% trường hợp file chung nằm ở chỗ quen của **chính các user sở hữu nó**, so với 23.4% với người lạ (mức nền). 558/821 file quen với mọi user đo được, 71 file chỉ quen với một phần nhóm |
| File chung không quen với ai | 170 file. Trung vị cách Hải Điến 807 km, so với 2.1 km của file chung "quen"; 110 file cách xa hơn 50 km. Dài trung vị 4.9 h, so với 0.6 h của file chung "quen". Đây là **những chuyến đi xa** |
| Ghi bằng hai máy riêng mà vẫn ở cạnh nhau | Trong các cặp có ≥ 30 phút ghi trùng nhau: 39% cặp có file chung ở cạnh nhau ≥ một nửa thời gian (7/18 cặp), so với **1.1%** ở các cặp không có file chung (1,596 cặp). Ví dụ 153/163 ở cạnh nhau 88% trong 3,072 phút, 126/167 là 92% trong 3,918 phút, 096/179 là 74%, 089/144 là 61%. Ngược lại 000/003 chỉ 1%, 128/153 là 2%, 084/085 là 2.5% |
| Tác động lên stay-point | 1,758/18,597 stay-point (9.5%) đến từ file chung, ở 48 user. Với user 124 và 156 là 100% thời gian ở lại, với 075 là 85% |

**Suy luận (chưa kiểm chứng được, vì không có thông tin về người dùng):**
1. **Phần lớn là đi cùng nhau thật.** Có ba bằng chứng: file chung nằm ở chỗ quen của chính các user sở hữu; nhiều file là chuyến đi xa kéo dài nhiều giờ, giống du lịch hay công tác theo nhóm; và các cặp lớn (153/163, 126/167) ở cạnh nhau gần như mọi lúc **kể cả khi ghi bằng hai máy riêng**. Nhiều khả năng là đồng nghiệp hoặc người sống cùng nhau.
2. **Một số cặp chỉ chung vài sự kiện.** 000/003, 128/153 và 084/085 có file chung nhưng gần như không bao giờ ở cạnh nhau khi ghi riêng. Có thể đó là vài chuyến đi chung, hoặc là log bị chép mà không đi cùng. Dữ liệu không phân biệt được hai khả năng này.
3. **Ba user 057, 094, 150 rất có thể là một nguồn dữ liệu được giao cho ba tài khoản.** Họ có toàn bộ dữ liệu giống hệt nhau và không có dữ liệu riêng nào. Có thể đó là một thiết bị ghi cho cả nhóm, hoặc là đăng ký trùng.
4. **Người có nhãn có thể là người ghi gốc.** 088 có nhãn chuyến đi và toàn bộ dữ liệu nằm trong 011, vốn không có nhãn. 075 có nhãn và gần như trùng hoàn toàn với 055, cũng không có nhãn. Người tự ghi nhãn cho chuyến đi thường là người cầm máy.

**Điều có thể xấu đi nếu không xử lý:**
- **Đếm trùng khi đánh giá.** Ba user 057, 094, 150 sẽ cho ra ba kết quả Home/Office giống hệt nhau, nên mọi chỉ số tính theo user đều đếm cùng một bằng chứng ba lần. Thống kê toàn bộ dữ liệu cũng bị thổi phồng bởi 1.46 triệu điểm thừa.
- **Rò rỉ dữ liệu khi chia train/test theo user** (v2, checkpoint 2). Cùng một file có thể nằm ở cả tập train lẫn tập test, nên chỉ số đo được sẽ đẹp hơn thực tế.
- **Home/Office có thể là của người khác.** 11 user có toàn bộ dữ liệu từ log chung. Nếu một user không thật sự đi cùng chuyến đó, "nhà" suy ra cho họ lại là nhà của người cầm máy. Đó vừa là kết quả sai, vừa là một rủi ro quyền riêng tư: suy ra chỗ ở của một người dưới danh tính của người khác.
- **Hai user "độc lập" có thể không độc lập.** 153 và 163 có thể có cùng Home vì sống cùng nhau. Kết quả không sai, nhưng hai user này không nên được tính như hai phép thử độc lập.

**Gợi ý cho các giai đoạn sau (chưa áp dụng):**
- Khi đánh giá classifier hoặc chia train/test, **gộp các user có file chung thành một nhóm**, để một nhóm không nằm ở cả hai phía.
- Báo cáo thêm chỉ số trên tập chỉ giữ **một đại diện mỗi nhóm**.
- Đánh dấu các user có > 90% dữ liệu từ file chung là **"ít độc lập"** trong kết quả đánh giá.

Các bảng chi tiết nằm trong `tables/`: `shared_pairs.csv`, `shared_share_by_user.csv`, `shared_file_familiarity.csv`, `shared_file_location.csv`, `colocation_summary.csv`, `shared_pairs_colocation.csv`, `staypoint_impact.csv`.